# 18 — Restore Agent1 and resume a request
Step 2 adds PersistentAgent, a subclass of your unchanged Agent, and a SQLiteCheckpointStore. Saving is explicit. A checkpoint stores ConversationState plus both runner request texts and the fresh-request flag.

Parts A–C run without an LLM. Part D is a live check: paste its code into notebook 16 if that notebook already has the model loaded. Opening notebook 18 does not automatically share notebook 16's variables or model.


In [1]:
from pathlib import Path
import sys
from uuid import uuid4
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src/agent1").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Open this notebook inside your Agent1 project.")
sys.path.insert(0, str(project_root / "src"))
from agent1.persistent_agent import PersistentAgent
from agent1.persistence.checkpoint_store import SQLiteCheckpointStore
from agent1.conversation.context_builder import ContextBuilder
from agent1.analysis.action_plan import ActionPlan

database_path = project_root / "data/local/conversations.sqlite3"
store = SQLiteCheckpointStore(database_path)
conversation_id = "offline-resume-" + uuid4().hex


## A. Save a pending request
This fixture starts the existing multi-action runner with an explicit plan. Planning accuracy is not tested here. No tool should run until the order number is supplied.

In [2]:
offline_agent = PersistentAgent(
    ensemble_model=None, router=None, analyzer=None,
    context_builder=ContextBuilder(), checkpoint_store=store,
)
request = "Track my order and check whether the blue backpack is available."
plan = ActionPlan.model_validate({
    "request_type": "multi_action", "reason": "Persistence fixture",
    "steps": [
        {"intent": "track_order", "action": "order_service", "arguments": {}},
        {"intent": "availability", "action": "product_service",
         "arguments": {"product": "backpack", "color": "blue"}},
    ],
})
offline_agent.state.add_user_message(request)
print(offline_agent.multi_action_runner.start(plan, request))
assert offline_agent.state.get_missing_arguments() == ["0.order_id"]
assert offline_agent.multi_action_runner.last_results == []
offline_agent.save_conversation(conversation_id)
print("Saved:", conversation_id)


For task 1 (track_order), please provide one order number, such as ORD-123.
Saved: offline-resume-328afaecc3504abdb13f3c8c84c8a26c


## B. Restore and complete it in a separate Python process
A bare order number follows Agent's existing pending-argument path and needs no model. The real mock tools run in the new process; completion is saved to the same ID.

In [3]:
import os, subprocess
child_code = """
import sys
from agent1.persistent_agent import PersistentAgent
from agent1.persistence.checkpoint_store import SQLiteCheckpointStore
from agent1.conversation.context_builder import ContextBuilder
agent = PersistentAgent(None, None, None, ContextBuilder(),
    checkpoint_store=SQLiteCheckpointStore(sys.argv[1]))
agent.load_conversation(sys.argv[2])
assert agent.state is agent.action_runner.state
assert agent.state is agent.multi_action_runner.state
assert agent.state.get_missing_arguments() == ['0.order_id']
assert agent.multi_action_runner.last_results == []
print(agent.handle_message('ORD-123'))
results = agent.multi_action_runner.last_results
assert len(results) == 2
assert all(r['result']['success'] for r in results)
assert results[0]['arguments'] == {'order_id': 'ORD-123'}
assert results[1]['result']['data']['quantity'] == 5
assert not agent.state.has_pending_action()
agent.save_conversation(sys.argv[2])
print('Separate-process Agent resume: PASS')
"""
env = os.environ.copy()
env["PYTHONPATH"] = str(project_root / "src") + os.pathsep + env.get("PYTHONPATH", "")
check = subprocess.run([sys.executable, "-c", child_code, str(database_path), conversation_id],
                       env=env, capture_output=True, text=True)
print(check.stdout)
if check.returncode:
    print(check.stderr)
    raise RuntimeError("Separate-process resume failed.")


1. Order ORD-123: shipped. Estimated delivery: 2026-09-23.
2. Mock inventory: blue backpack: 5 available.
Separate-process Agent resume: PASS



## C. Reopen the completed checkpoint; loading executes nothing

In [4]:
finished = PersistentAgent(None, None, None, ContextBuilder(), checkpoint_store=store)
finished.load_conversation(conversation_id)
assert not finished.state.has_pending_action()
assert finished.multi_action_runner.last_results == []
tool_messages = [m for m in finished.state.messages if m.role.value == "tool"]
assert len(tool_messages) == 2
print("Stored tool results:", len(tool_messages))
print("Saved completion restored: PASS")
# The old in-memory instance has not changed; do not save it over the completed row.
assert offline_agent.state.has_pending_action()


Stored tool results: 2
Saved completion restored: PASS


## D. Live check with your already-loaded model
Paste the code below into notebook 16, after its source-backed setup. It is intentionally Markdown here so Run All does not unexpectedly load or require a model. The same code is in `18_live_resume_cell.py`.

```python
# Run in the notebook/kernel where your working `agent` is already loaded.
# No model load, training, or analyzer monkey-patching occurs here.
from pathlib import Path
from uuid import uuid4
from agent1.persistent_agent import PersistentAgent
from agent1.persistence.checkpoint_store import SQLiteCheckpointStore

project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "src/agent1").is_dir())
checkpoint_store = SQLiteCheckpointStore(project_root / "data/local/conversations.sqlite3")
template_agent = agent

def new_persistent_agent():
    return PersistentAgent(
        ensemble_model=template_agent.ensemble_model,
        router=template_agent.router,
        analyzer=template_agent.analyzer,
        context_builder=template_agent.context_builder,
        top_n_intents=template_agent.top_n_intents,
        response_service=template_agent.response_service,
        checkpoint_store=checkpoint_store,
    )

conversation_id = "live-resume-" + uuid4().hex
before = new_persistent_agent()
print("Before save:", before.handle_message(
    "Track my order and check whether the blue backpack is available."))
assert before.state.has_pending_action()
assert len(before.state.pending_action_plan.steps) == 2
assert before.state.get_missing_arguments() == ["0.order_id"]
assert before.multi_action_runner.last_results == []
before.save_conversation(conversation_id)

resumed = new_persistent_agent()
resumed.load_conversation(conversation_id)
assert resumed.state is resumed.action_runner.state
assert resumed.state is resumed.multi_action_runner.state
assert resumed.state.messages == before.state.messages
assert resumed.multi_action_runner.last_results == []
print("Restored missing:", resumed.state.get_missing_arguments())
print("After restore:", resumed.handle_message("ORD-123"))
records = resumed.multi_action_runner.last_results
assert [(r["intent"], r["arguments"]) for r in records] == [
    ("track_order", {"order_id": "ORD-123"}),
    ("availability", {"product": "backpack", "color": "blue"}),
]
assert all(r["result"]["success"] for r in records)
assert not resumed.state.has_pending_action()
# Save completion under the SAME ID, replacing the pending checkpoint.
resumed.save_conversation(conversation_id)

completed = new_persistent_agent().load_conversation(conversation_id)
assert not completed.state.has_pending_action()
assert completed.state.messages == resumed.state.messages
assert completed.multi_action_runner.last_results == []
print("SAVE / RESTORE / RESUME: PASS")
print("Review the answer: shipped ORD-123 and 5 blue backpacks (mock inventory).")
print("Saved conversation:", conversation_id)

```

## Boundaries
- Save after each turn you want to preserve. There is no autosave.
- Save again after execution, under the same ID, to replace the pending checkpoint.
- Restoring an older pending checkpoint and replying again can repeat those read-only lookups. This is not exactly-once crash recovery.
- The inherited reset() clears in-memory state; it does not delete saved rows.
- Loading replaces the instance's current conversation. Use a new instance, or save the previous conversation first.
- Reuse model objects sequentially; this notebook does not establish thread-safe concurrent model access.
- Models, classifier, knowledge files and code must be configured again after a real process/kernel restart. Only conversation/runtime state is stored.
- Earlier step-1 snapshots remain separate. They lack runner runtime and are not automatically restored as Agent checkpoints.
